# Data Splitting — By Repetition Index

**Key objectives:**
1. **Repetition-based split**: repetition 1, 2, 3 go to train; repetition 4 goes to val; repetition 5 goes to test.
2. **All speakers present in every split**: every penutur appears across train, val, and test because the split is done by repetition, not by speaker.
3. **No repetition leakage**: a repetition index must not appear in more than one split.

In [1]:
import pickle
import pandas as pd
import os


## 1. Load Excel & Mapping


In [2]:
excel_path = "./raw_data/Gloss dan Tanda Dataset.xlsx"
df_excel = pd.read_excel(excel_path)
display(df_excel.head())

# TODO: Adjust column names to match the actual Excel file headers
col_sentence = 'ID'       # e.g. 'ID Tanda' or 'Sentence ID'
col_gloss    = 'Gloss'    # gloss label column
col_text     = 'Kalimat'  # natural language sentence column

mapping      = {}
mapping_text = {}

if col_sentence in df_excel.columns and col_gloss in df_excel.columns:
    for _, row in df_excel.dropna(subset=[col_sentence]).iterrows():
        key = str(row[col_sentence]).strip()
        mapping[key] = str(row[col_gloss]).strip()
        if col_text in df_excel.columns:
            mapping_text[key] = str(row[col_text]).strip()
else:
    print("WARNING: Column not found. Check col_sentence and col_gloss variable names.")


,No.,ID,Kalimat,Gloss,N Gloss,Tanda-1,Tanda-2,Tanda-3,Tanda-4,Tanda-5,Tanda-6,Tanda-7,Tanda-8,Tanda-9,N Tanda
0,1.0,S001,Di mana ayah sama Ibu?,AYAH SAMA IBU MANA,4.0,AYAH,SAMA,IBU,MANA,NaN,NaN,NaN,NaN,NaN,4.0
1,2.0,S002,Dia anak baik sehingga banyak orang menyukainya,DIA ANAK BAIK SAMPAI BANYAK ORANG SUKA,7.0,DIA,ANAK,BAIK,SAMPAI,BANYAK,ORANG,SUKA,NaN,NaN,7.0
2,3.0,S003,Apa kamu pernah membaca novel bahasa inggris?,APA KAMU PERNAH BACA NOVEL B.INGGRIS,6.0,APA,KAMU,PERNAH,BACA,NOVEL,B. INGGRIS,NaN,NaN,NaN,6.0
3,4.0,S004,"Badan ku gemuk, tapi badan adik kurus",BADAN AKU GEMUK TAPI BADAN ADIK KURUS,7.0,BADAN,AKU,GEMUK,TAPI,BADAN,ADIK,KURUS,NaN,NaN,7.0
4,5.0,S005,"Saya sering pusing, saya harus periksa ke mana?","AKU PUSING SERING, AKU HARUS PERIKSA MANA",7.0,AKU,PUSING,SERING,AKU,HARUS,PERIKSA,MANA,NaN,NaN,7.0


## 2. Load IDs from Pickle


In [3]:
pickle_path = "../data/pickle/pose_bisindo.pkl"

with open(pickle_path, 'rb') as f:
    data = pickle.load(f)

all_ids = sorted(data.keys())
print(f"Total unique IDs found: {len(all_ids)}")
print("Sample IDs:", all_ids[:5])


Total unique IDs found: 750
Sample IDs: ['P01_S001_R01', 'P01_S001_R02', 'P01_S001_R03', 'P01_S001_R04', 'P01_S001_R05']


## 3. Build DataFrame


In [ ]:
records = []
for file_id in all_ids:
    parts = file_id.split('_')
    if len(parts) != 3:
        continue

    signer, sentence, repetition = parts
    repetition = int(str(repetition).lstrip('Rr'))

    gloss = mapping.get(sentence,      f"UNKNOWN_{sentence}")
    text  = mapping_text.get(sentence, f"UNKNOWN_{sentence}")

    records.append({
        'id'        : file_id,
        'gloss'     : gloss,
        'text'      : text,
        'signer'    : signer,
        'sentence'  : sentence,
        'repetition': repetition,
    })

df = pd.DataFrame(records)
display(df.head())

ValueError: invalid literal for int() with base 10: 'R01'

## 4. Repetition-Based Split Algorithm

Split the dataset by repetition index so that all speakers remain distributed across every split. This matches the intended 5 repetition setup: repetitions 1-3 for train, repetition 4 for validation, and repetition 5 for test.

In [ ]:
train_repetitions = {1, 2, 3}
dev_repetitions   = {4}
test_repetitions  = {5}

train_ids = df[df['repetition'].isin(train_repetitions)]['id'].tolist()
dev_ids   = df[df['repetition'].isin(dev_repetitions)]['id'].tolist()
test_ids  = df[df['repetition'].isin(test_repetitions)]['id'].tolist()

# Build final split DataFrames
df_train = df[df['id'].isin(train_ids)].copy()
df_dev   = df[df['id'].isin(dev_ids)  ].copy()
df_test  = df[df['id'].isin(test_ids) ].copy()

print(f"Total samples : {len(df)}")
print(f"Train         : {len(df_train)} ({len(df_train)/len(df):.1%})")
print(f"Dev           : {len(df_dev)}   ({len(df_dev)/len(df):.1%})")
print(f"Test          : {len(df_test)}  ({len(df_test)/len(df):.1%})")

print("\nRepetition distribution:")
print("Train repetitions:", sorted(df_train['repetition'].unique().tolist()))
print("Dev repetitions  :", sorted(df_dev['repetition'].unique().tolist()))
print("Test repetitions :", sorted(df_test['repetition'].unique().tolist()))

Total samples : 750
Train         : 450 (60.0%)
Dev           : 150   (20.0%)
Test          : 150  (20.0%)


## 5. Validation (Must Pass)

Two hard checks are enforced before saving any output:
- **Coverage**: every gloss class must appear in all three splits.
- **Repetition separation**: train, dev, and test must each contain only the repetition indices assigned to them.

In [ ]:
# ── Coverage check ───────────────────────────────────────────────────────────
train_glosses = set(df_train['gloss'])
dev_glosses   = set(df_dev['gloss'])
test_glosses  = set(df_test['gloss'])
all_glosses   = set(df['gloss'])

print("--- Coverage Validation (100% of gloss classes must be present) ---")
assert train_glosses == all_glosses, f"Train missing glosses: {all_glosses - train_glosses}"
assert dev_glosses   == all_glosses, f"Dev missing glosses: {all_glosses - dev_glosses}"
assert test_glosses  == all_glosses, f"Test missing glosses: {all_glosses - test_glosses}"
print("PASSED: All gloss classes present in Train, Dev, and Test.")

# ── Repetition separation check ──────────────────────────────────────────────
train_reps = set(df_train['repetition'])
dev_reps   = set(df_dev['repetition'])
test_reps  = set(df_test['repetition'])

print("\n--- Repetition Validation (each split gets its assigned repetitions only) ---")
assert train_reps == {1, 2, 3}, f"Train repetitions mismatch: {train_reps}"
assert dev_reps   == {4},       f"Dev repetitions mismatch: {dev_reps}"
assert test_reps  == {5},       f"Test repetitions mismatch: {test_reps}"
print("PASSED: Repetition-based split is correct.")

# Optional speaker coverage sanity check
all_signers = set(df['signer'])
assert set(df_train['signer']) == all_signers, "Train does not contain all signers"
assert set(df_dev['signer'])   == all_signers, "Dev does not contain all signers"
assert set(df_test['signer'])  == all_signers, "Test does not contain all signers"
print("PASSED: All signers appear in Train, Dev, and Test.")

--- Coverage Validation (100% of gloss classes must be present) ---
PASSED: All gloss classes present in Train, Dev, and Test.

--- Leakage Validation (no signer-sentence group may overlap across splits) ---
PASSED: Zero data leakage — no group appears in more than one split.


## 6. Export CSV


In [ ]:
output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

cols_id_gloss = ['id', 'gloss']
cols_full     = ['id', 'gloss', 'text']

# Primary split files (id | gloss)
df_train[cols_id_gloss].to_csv(os.path.join(output_dir, 'train.csv'), index=False, sep='|')
df_dev  [cols_id_gloss].to_csv(os.path.join(output_dir, 'dev.csv'),   index=False, sep='|')
df_test [cols_id_gloss].to_csv(os.path.join(output_dir, 'test.csv'),  index=False, sep='|')

# Extended list files (id | gloss | text)
df_train[cols_full].to_csv(os.path.join(output_dir, 'sd_train_list.txt'), index=False, sep='|')
df_dev  [cols_full].to_csv(os.path.join(output_dir, 'sd_dev_list.txt'),   index=False, sep='|')
df_test [cols_full].to_csv(os.path.join(output_dir, 'sd_test_list.txt'),  index=False, sep='|')

print(f"Done. Output saved to '{output_dir}/'.")


Done. Output saved to 'results/'.
